In [100]:
import pandas as pd

## 1. Load and Inspect the Dataset

The first step is to load the Excel dataset and understand its structure, dimensions, column names, and data types.


In [ ]:
df = pd.read_excel('Customer_Call_List.xlsx')
print("Dataset Shape:", df.shape)
df.info()

Dataset Shape: (21, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   CustomerID         21 non-null     int64 
 1   First_Name         21 non-null     object
 2   Last_Name          20 non-null     object
 3   Phone_Number       19 non-null     object
 4   Address            21 non-null     object
 5   Paying Customer    21 non-null     object
 6   Do_Not_Contact     17 non-null     object
 7   Not_Useful_Column  21 non-null     bool  
dtypes: bool(1), int64(1), object(6)
memory usage: 1.3+ KB


## 2. Remove Unnecessary Columns

In [102]:
if 'Not_Useful_Column' in df.columns:
    df = df.drop(columns='Not_Useful_Column')

In [103]:
df.head()

,CustomerID,First_Name,Last_Name,Phone_Number,Address,Paying Customer,Do_Not_Contact
0,1001,Frodo,Baggins,123-545-5421,"123 Shire Lane, Shire",Yes,No
1,1002,Abed,Nadir,123/643/9775,93 West Main Street,No,Yes
2,1003,Walter,/White,7066950392,298 Drugs Driveway,N,NaN
3,1004,Dwight,Schrute,123-543-2345,"980 Paper Avenue, Pennsylvania, 18503",Yes,Y
4,1005,Jon,Snow,876|678|3469,123 Dragons Road,Y,No


In [104]:
df = df.drop_duplicates()

## 3. Clean Customer Names

Some customer last names contain unwanted leading or trailing characters.

String operations are used to remove these characters and standardize the values.


In [105]:
df['Last_Name'] = (
    df['Last_Name']
    .astype(str)
    .str.strip()
    .str.strip('123._/'))


## 4. Removing non numeric charecters from Phone Number column

In [106]:
df['Phone_Number'] = (
    df['Phone_Number']
    .astype(str)
    .str.replace(r'\D','',regex=True)
    )
df['Phone_Number'] = df['Phone_Number'].replace('',pd.NA)
df[['CustomerID','Phone_Number']].head(10)


,CustomerID,Phone_Number
0,1001,1235455421
1,1002,1236439775
2,1003,7066950392
3,1004,1235432345
4,1005,8766783469
5,1006,3047622467
6,1007,<NA>
7,1008,8766783469
8,1009,<NA>
9,1010,1235455421


## 5. Split Address Information

The original `Address` field contains multiple pieces of information.

Where applicable, the address is separated into:

* `Street_Address`
* `State`
* `Zip_Code`

This creates a more structured dataset for downstream analysis.


In [107]:
df[['Street_Address','State','Zip_Code']] = df["Address"].str.split(',',expand=True)
df[['Street_Address','State','Zip_Code']].head(10)

,Street_Address,State,Zip_Code
0,123 Shire Lane,Shire,None
1,93 West Main Street,None,None
2,298 Drugs Driveway,None,None
3,980 Paper Avenue,Pennsylvania,18503
4,123 Dragons Road,None,None
5,768 City Parkway,None,None
6,1209 South Street,None,None
7,98 Clue Drive,None,None
8,123 Middle Earth,None,None
9,25th Main Street,New York,None


## 6. Standardize categorical columns

In [108]:
df['Do_Not_Contact'] = (
    df['Do_Not_Contact']
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        'YES': 'Y',
        'NO': 'N'
        })
)
df['Paying Customer'] = (
    df['Paying Customer']
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        'YES':'Y', 
        'NO':'N'
        })
)
print(df['Do_Not_Contact'].value_counts(dropna=False))
print(df['Paying Customer'].value_counts(dropna=False))

Do_Not_Contact
N      12
Y       4
NAN     4
Name: count, dtype: int64
Paying Customer
Y      13
N       6
N/A     1
Name: count, dtype: int64


## 4. Handle Missing Values

The dataset contains both `N/a` and missing (`NaN`) values.

These values are standardized to Pandas missing values so that missing-data patterns can be identified consistently.


In [109]:
df = df.replace('N/a', pd.NA)
df.isna().sum()

CustomerID          0
First_Name          0
Last_Name           0
Phone_Number        4
Address             1
Paying Customer     0
Do_Not_Contact      0
Street_Address      1
State              14
Zip_Code           19
dtype: int64

## 5. Apply the business rules using vectorized Pandas filtering

In [110]:
df = df[df['Do_Not_Contact'] != 'Y']
df = df[df['Phone_Number'].notna()]

## 6. Remove Duplicates

In [111]:
df = df.drop_duplicates();

## 7. Reset index

In [112]:
df = df.reset_index(drop=True)

## 8. Before

In [113]:
initial_rows = len(df)
initial_columns = len(df.columns)
print('Initial rows:', initial_rows)
print('Initial columns:',initial_columns)

Initial rows: 13
Initial columns: 10


## 9. Aafter duplicates removed

In [114]:
duplicates_removed = initial_rows - len(df)

print("Duplicates removed:", duplicates_removed)

Duplicates removed: 0


In [115]:
print("Final rows:", len(df))
print("Final columns:", len(df.columns))

Final rows: 13
Final columns: 10


In [116]:
df.isna().sum()

CustomerID          0
First_Name          0
Last_Name           0
Phone_Number        0
Address             0
Paying Customer     0
Do_Not_Contact      0
Street_Address      0
State               9
Zip_Code           13
dtype: int64

## 10 Final Data Qulaity Check

In [117]:
print("Final dataset shape:", df.shape)

print("\nDuplicate rows:", df.duplicated().sum())

print("\nMissing values:")
print(df.isna().sum())

print("\nPaying Customer values:")
print(df['Paying Customer'].value_counts(dropna=False))

print("\nDo_Not_Contact values:")
print(df['Do_Not_Contact'].value_counts(dropna=False))

Final dataset shape: (13, 10)

Duplicate rows: 0

Missing values:
CustomerID          0
First_Name          0
Last_Name           0
Phone_Number        0
Address             0
Paying Customer     0
Do_Not_Contact      0
Street_Address      0
State               9
Zip_Code           13
dtype: int64

Paying Customer values:
Paying Customer
Y    9
N    4
Name: count, dtype: int64

Do_Not_Contact values:
Do_Not_Contact
N      10
NAN     3
Name: count, dtype: int64
